In [1]:
%run 00_config.py

In [20]:
from pymilvus import MilvusClient, DataType
schema = MilvusClient.create_schema()
from pymilvus import CollectionSchema, FieldSchema, DataType
from pymilvus.bulk_writer import BulkFileType


##### In this notebook we will create index image + text vector embeddings

- Embeddings will be created using Visual BGE model
- We will store those vector embeddings on the minio container which emulates cloud object storage like S3
- Products have between 0 - 7 images. we will only index the first 5.
- Products with any missing images will be skipped
    * We skip products with missing images because the first 5 are indexed. If any of number 1-5 are missing, reading them would throw an error

In [23]:
schema = CollectionSchema(fields=[
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="product_id", dtype=DataType.VARCHAR, max_length=32),
    FieldSchema(name="vector", dtype=DataType.FLOAT_VECTOR, dim=768),
    FieldSchema(name="prod_imagenum", dtype=DataType.INT32),
    FieldSchema(name="image_file", dtype=DataType.VARCHAR, max_length=256),
    FieldSchema(name="prod_text", dtype=DataType.VARCHAR, max_length=1000),
    ]
)

In [22]:
import json
#do this for the very first time.
# am_products= load_dataset("bprateek/amazon_product_description")
# am_products = am_products['train']
am_products = []
with open('product-train.jsonl') as f:
    for line in f:
        am_products.append(json.loads(line))

In [24]:
from encoder_visual_bge import Encoder

def initialize():
    model_name = "BAAI/bge-base-en-v1.5"
    model_path = "models/Visualized_base_en_v1.5.pth"  # Change to your own value if using a different model path
    encoder = Encoder(model_name, model_path)
    return encoder

encoder = initialize()

In [25]:
def get_entity(image_path: str, product_img_text: str, product_id: str, prod_imagenum: int) -> dict:
    """
    Get the entity for the given image and text.
    """
    # Encode the image and text
    vector = encoder.encode_image(f"{image_path}", product_img_text)

    data = {
        #"id": None,  # Auto-generated ID
        "product_id": product_id,
        "vector": vector,
        "prod_imagenum": prod_imagenum,
        "image_file": image_path,
        "prod_text": product_img_text
    }

    return data

In [ ]:
import os

image_and_text_args = []
for i in range(21):  # Adjust the range as needed. There are 10002 products
    print(f"iteration: {i}")
    for product in am_products[i*500: (i+1)*500]:  # Adjust the range as needed
        # get unique product ID from the product
        product_id = product.get('Uniq Id', '')

        # set category to Unknown if it's not in the data
        category = product.get('Category', 'Unknown') if product.get('Category') is not None else 'Unknown'

        product_img_text = f"{product.get('Product Name', '') } sold in these categories: [{",".join(category.split('|'))}]"
        # now handle images
        all_images_urls = list(enumerate(product.get('Image', '').split('|')))
        
        image_file_paths = list(map(lambda x, id=product_id: local_img_path(x, id), all_images_urls)) # construct the expected local image paths
        if len(all_images_urls) > 0:                                                # for products with at least one image URL
            if "transparent-pixel" in image_file_paths[-1].lower():
                image_file_paths = image_file_paths[:-1]
            valid_images_bool = [1 if os.path.isfile(f"{img_path}") else 0 for img_path in image_file_paths]
            if not any(valid_images_bool):
                all_images_urls = [url for idx, url in all_images_urls]
                #print(f"All images are invalid for product {product.get('Product Name')}, {", ".join(all_images_urls)}")
                pass
            if all(valid_images_bool):
                for prod_imaegnum, img_path in enumerate(image_file_paths[:5]):  # limit to first 5 images only, the remaining are for testing         
                    image_and_text_args.append((f"{img_path}", product_img_text, product_id, prod_imaegnum+1))



In [ ]:
total = len(image_and_text_args)
print(f"Total images to process: {total}")
total_distinct_products = len(set([args[2] for args in image_and_text_args]))
print(f"Total distinct products with images: {total_distinct_products}")
import random
random_elements = random.sample(image_and_text_args, 10)
print("Random sample of 10 image-text arguments:") 
for elem in random_elements:
    print(elem)

In [ ]:
from pymilvus.bulk_writer import RemoteBulkWriter
MINIO_ADDRESS = "localhost:9000"
MINIO_SECRET_KEY = "minioadmin"
MINIO_ACCESS_KEY = "minioadmin"
writer = RemoteBulkWriter(
        schema=schema,
        remote_path="/", # data files will be written to this path on minio
        chunk_size=64*1024*1024, # expected size of each data file
        connect_param=RemoteBulkWriter.S3ConnectParam(
            endpoint=MINIO_ADDRESS, # the address ought to be the minio server the milvus is using
            access_key=MINIO_ACCESS_KEY,
            secret_key=MINIO_SECRET_KEY,
            bucket_name="a-bucket", # this bucket ought to be the bucket the milvus server is using, so that milvus can access the uploaded data files
        ),
        file_type=BulkFileType.PARQUET,
)

##### This ends up taking anywhere from 30-45 minutes

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import concurrent.futures as cf

thread_pool_size = os.cpu_count() + 4
pcount = 0
with ThreadPoolExecutor(max_workers=thread_pool_size) as executor:
    #print("adding initial future ", x, image_and_text_args[x][0])
    futures = [executor.submit(get_entity, *x) for x in image_and_text_args]
    for future in cf.as_completed(futures):
        try:
            result = future.result()
            writer.append_row(result)
            pcount += 1
            if(pcount % 3000 == 0):
                writer.commit()
                print("committed at product count ", pcount)
        except Exception as e:
            print(f"Task raised an exception: {e}")
    writer.commit()
    print("Final commit at product count ", pcount)

##### We now print all of the filenames we produced using _writer.commit_

- if you are running milvus using docker-compose you can see them on http://localhost:9000 using minioadmin:minioadmin

In [31]:
output_files = writer.batch_files
print(output_files)

[['21626c86-6da8-49d8-8720-4d4b59f97961/1.parquet'], ['21626c86-6da8-49d8-8720-4d4b59f97961/2.parquet'], ['21626c86-6da8-49d8-8720-4d4b59f97961/3.parquet'], ['21626c86-6da8-49d8-8720-4d4b59f97961/4.parquet'], ['21626c86-6da8-49d8-8720-4d4b59f97961/5.parquet'], ['21626c86-6da8-49d8-8720-4d4b59f97961/6.parquet'], ['21626c86-6da8-49d8-8720-4d4b59f97961/7.parquet'], ['21626c86-6da8-49d8-8720-4d4b59f97961/8.parquet'], ['21626c86-6da8-49d8-8720-4d4b59f97961/9.parquet'], ['21626c86-6da8-49d8-8720-4d4b59f97961/10.parquet']]


In [32]:
client = MilvusClient(
    uri="http://localhost:19530",
    token="root:Milvus"
)

In [33]:
client.drop_collection(collection_name, database_name="default")

In [34]:
client.create_collection(
    collection_name=collection_name,
    schema=schema,
    database_name="default")
coll_metadata = [{
    "collection_name": collection_name,
    "vector_field_name": "vector",
    "dimension": 768,
    "primary_field_name": "id",
    "schema": -1,
    "index_params": [
        {
            "index_name": "vector_index",
            "field_name": "vector",
            "index_type": "AUTOINDEX",
            "metric_type": "COSINE",
        },
        {
            "index_name": "product_id_index",
            "field_name": "product_id",
            "index_type": "INVERTED",
        }, 
        {
            "index_name": "prod_imagenum_index",
            "field_name": "prod_imagenum",
            "index_type": "INVERTED",
        }, 
        {
            "index_name": "prod_text_index",
            "field_name": "prod_text",
            "index_type": "INVERTED",
        }
        ]
    },
]
# Create the collection
index_params = MilvusClient.prepare_index_params()
for index_param in coll_metadata[0]["index_params"]:
    index_params.add_index(index_name=index_param["index_name"], 
                            field_name=index_param["field_name"], 
                            index_type=index_param["index_type"], 
                            metric_type=index_param.get("metric_type", None))
client.create_index(
    collection_name=coll_metadata[0]["collection_name"],
    index_params=index_params,
    database_name="default"
)
client.flush(
    collection_name=collection_name,
    database_name="default"
)

In [35]:
import json
from pymilvus.bulk_writer import bulk_import
resp = bulk_import(
    url = "http://localhost:19530",
    db_name="default",
    collection_name = collection_name,
    files = writer.batch_files
)
    

In [ ]:
job_id = resp.json()['data']['jobId']
print(job_id)

In [ ]:
from pymilvus.bulk_writer import get_import_progress
url = f"http://127.0.0.1:19530"

resp = get_import_progress(
    url=url,
    job_id=job_id,
)

print(json.dumps(resp.json(), indent=4, sort_keys=True))

##### Expect 29,573 total records (aka. entities) after the import process

In [ ]:
client.get_collection_stats(
    collection_name=collection_name)

In [21]:
client.load_collection(
    collection_name=collection_name,
    database_name="default"
)

In [22]:
result = client.query(
    collection_name=collection_name,
    filter="RANDOM_SAMPLE(0.1)",
    output_fields=["image_file", "prod_text"],
    limit=5,
)

In [ ]:
print(result)